# ROSP-7: System delivered image quality (sDIQ)

PTH Tier 1 metric **ROSP-7** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-7](https://rubinobs.atlassian.net/browse/ROSP-7)
- Confluence metric page: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906933804](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906933804)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Elana Urbach |
| Owner | Sandrine Thomas |
| Category | 2 - Data Production |
| Layer | 2 |

Method version: `rosp-07-v1.0` (draft until the tag is cut; the frozen commit SHA is recorded on the Confluence page). Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

**Method.** The system contribution to the delivered image quality is what is left of the delivered PSF FWHM
once the atmosphere is removed in quadrature. For every LSSTCam science visit of the selected programs:

* the **delivered PSF FWHM** is `psf_sigma_median` × 2√(2 ln 2) × 0.2″/pixel from `cdb_lsstcam.visit1_quicklook`,
  exactly as in ROSP-2 and **as delivered** (no zenith / 500 nm correction);
* the **atmospheric seeing** is taken from an independent monitor and expressed in the same frame as the PSF,
  i.e. at the airmass $X$ and band of the visit. Monitors that report at zenith and 500 nm are de-corrected with the
  inverse of the ROSP-2 correction, $\theta_\mathrm{obs} = \theta_\mathrm{z,500}\,X^{0.6}\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{-0.2}$,
  with the same effective wavelengths as ROSP-2. Three monitors are used, each giving its own sDIQ:
  1. **RINGSS** scintillation seeing (`fwhmScintillation` of `lsst.sal.ESS.logevent_ringssMeasurement`, salIndex 304): the median of the
     EFD measurements within ± `ringss_window_minutes` of the visit midpoint, or, with `ringss_source = consdb`, the per-visit
     mean already in the ConsDB (`efd_lsstcam.visit1_efd.mt_ringss_fwhm_scintillation`). Reported at zenith / 500 nm, de-corrected.
  2. **DIMM** (`cdb_lsstcam.visit1.dimm_seeing`, Kolmogorov seeing at zenith / 500 nm) corrected for a finite outer scale $L_0$ = `dimm_outer_scale_m`
     (von Kármán, Tokovinin 2002), with the Fried parameter scaled to the visit **before** the correction:
     $r_0 = 0.98 \cdot 500\,\mathrm{nm} / \theta_\mathrm{DIMM}$, $r_0(\lambda, X) = r_0\,(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{1.2}\,X^{-0.6}$,
     $\theta_\mathrm{obs} = \dfrac{0.98\,\lambda_\mathrm{eff}}{r_0(\lambda, X)}\sqrt{1 - 2.18\,(r_0(\lambda, X)/L_0)^{0.356}}$.
     The Kolmogorov part of this is the same $X^{0.6}(\lambda_\mathrm{eff}/500\,\mathrm{nm})^{-0.2}$ de-correction as for RINGSS; the difference is that the
     outer-scale factor is evaluated at the band and airmass of the visit rather than at zenith / 500 nm (it is a stronger reduction in the red). The DIMM is only used on visits
     with an outside wind speed (`visit1.wind_speed`) of at least `dimm_min_wind_speed` = {{ params.dimm_min_wind_speed }} m/s, below which it is known to be unreliable.
  3. **Guider** free-atmosphere seeing: the mid (0.5 - 1.5 km) and high (> 1.5 km) layers of the guider tomographic
     seeing, $\sqrt{\mathtt{guider\_mid\_layer\_seeing}^2 + \mathtt{guider\_free\_seeing}^2}$ from `visit1_quicklook`, without the ground layer.
     The producer (`lsst.summit.utils.guiders.seeing`) derives these from the guider-star centroid motion along the line of sight and scales
     them to the exposure wavelength, so they are already in the frame of the visit and are used as they are.

Per visit the squared system term is $\mathrm{sys}^2 = \mathrm{FWHM}^2 - \theta_\mathrm{atm,obs}^2$ (arcsec²), which is noisy and may be
negative. The metric for any set of visits is

$$ \mathrm{sDIQ} = \sqrt{\operatorname{median}(\mathrm{sys}^2)}, $$

the median taken before the square root so that negative visits are kept. A negative median is shown as the signed
square root. The error quoted on sDIQ is the robust standard error of the median of $\mathrm{sys}^2$ (1.2533 × 1.4826 MAD / √n) propagated through the square root.

**Selection.** The ROSP-2 selection: visits need a PSF measurement, at least {{ params.min_psf_stars }} PSF stars
(`n_psf_star_total`, the `min_psf_stars` parameter), an airmass and a band, and `filter_change_close_loop` visits are dropped.
Each monitor's sDIQ then uses the selected visits that have a finite, positive value from that monitor; the only monitor-specific cut is the DIMM wind-speed one.

Everything is given per monitor, for all bands together and per band: the sDIQ with its error and the spread of $\mathrm{sys}^2$ (table),
night by night (nightly sDIQ with the 25-75 % range of the per-visit term) and as histograms of $\mathrm{sys}^2$.

The headline values are the all-band sDIQ for each of the three monitors.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch and the standard science blocks.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>; empty = all programs
min_psf_stars = (
    50  # drop visits with fewer PSF stars (n_psf_star_total) than this; 0 = no cut
)
dimm_outer_scale_m = (
    30  # outer scale L0 (m) for the von Karman DIMM correction; 0 = no correction
)
dimm_min_wind_speed = (
    6.0  # DIMM only used on visits with wind_speed at or above this (m/s); 0 = no cut
)
ringss_source = "efd"  # "efd": EFD median within the window; "consdb": efd_lsstcam.visit1_efd per-visit mean
ringss_window_minutes = (
    5  # EFD RINGSS measurements within +/- this of the visit midpoint are medianed
)
efd_name = "usdf_efd"  # lsst_efd_client instance name
use_cache = False  # cache the ConsDB/EFD pulls as parquet under ./cache (ignored on Times Square)

not_times_square = True

In [ ]:
import inspect
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()
try:
    not_times_square
except NameError:  # Times Square replaced the parameters cell, which defines the flag
    not_times_square = False

# Bumped through the PTH Method Change Process; the baseline commit is tagged rosp-07-vX.Y.
METHOD_VERSION = "rosp-07-v1.0"

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)
REFERENCE_WAVELENGTH_NM = 500.0
EXCLUDE_REASONS = ("filter_change_close_loop",)
MIN_VISITS_PER_NIGHT = (
    5  # a monitor's nightly sDIQ needs at least this many valid visits
)
# Effective wavelengths (nm) copied from lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam (as ROSP-2);
# the Provenance cell at the end checks them against the kernel's summit_utils.
EFFECTIVE_WAVELENGTH_SOURCE = (
    "lsst.summit.utils.utils.getBandpassSeeingCorrection, summit_utils w.2026.44"
)
EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BANDS = ("u", "g", "r", "i", "z", "y")
BAND_COLORS = dict(
    zip(BANDS, ("#4a3aa7", "#2a78d6", "#1baf7a", "#eda100", "#eb6834", "#e34948"))
)
# The guider tomographic seeing (lsst.summit.utils.guiders.seeing) is measured along the line of sight and
# scaled to the exposure wavelength by its producer, so it is already in the frame of the visit.
GUIDER_SEEING_FRAME = (
    "observed"  # "zenith_500nm" if the producer ever reports at zenith / 500 nm
)
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "savefig.dpi": 150,
    }
)


@dataclass(frozen=True)
class Monitor:
    """One atmospheric seeing estimate: its input column, the frame that column is in, and plot style."""

    tag: str
    label: str
    source: str  # column holding the monitor value
    frame: str  # "zenith_500nm" (de-corrected to the visit) or "observed" (used as is)
    color: str
    marker: str

    @property
    def z500(self):
        return f"{self.tag}_zenith_500nm"

    @property
    def atm(self):
        return f"atm_{self.tag}"

    @property
    def sys2(self):
        return f"sys2_{self.tag}"


def make_monitors(L0, min_wind):
    dimm_label = f"DIMM, L0 = {L0:g} m" if L0 else "DIMM, no outer-scale correction"
    if min_wind > 0:
        dimm_label += f", wind >= {min_wind:g} m/s"
    return (
        Monitor(
            "ringss",
            "RINGSS scintillation",
            "ringss_z500",
            "zenith_500nm",
            "#2a78d6",
            "o",
        ),
        # the DIMM value is de-corrected through r0 (see below), so it arrives already in the visit frame
        Monitor("dimm", dimm_label, "dimm_obs", "observed", "#eb6834", "s"),
        Monitor(
            "guider",
            "guider mid + high layers",
            "guider_free_atm",
            GUIDER_SEEING_FRAME,
            "#1baf7a",
            "^",
        ),
    )


def parse_day_obs(value) -> int:
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """Comma/space separated block names; bare numbers become BLOCK-<n>; empty means no program cut."""
    items = [v for v in re.split(r"[,\s]+", str(value or "").strip()) if v]
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def parse_outer_scale(value):
    """Outer scale in metres, or None for no correction (0, empty, 'none', 'inf')."""
    s = str(value).strip().lower()
    if s in ("", "none", "inf", "nan", "0", "0.0"):
        return None
    L0 = float(s)
    return L0 if L0 > 0 else None


def day_obs_chunks(day_obs_min, day_obs_max, chunk_days=31):
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def day_obs_window(day_obs_lo, day_obs_hi):
    """UTC start/end (astropy Time, as lsst_efd_client requires) covering the nights day_obs_lo..day_obs_hi:
    12:00 UTC to 12:00 UTC the day after."""
    from astropy.time import Time

    t0 = datetime.strptime(str(day_obs_lo), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    )
    t1 = datetime.strptime(str(day_obs_hi), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    ) + pd.Timedelta(days=1)
    return Time(t0), Time(t1)


def _read_cache(cache_path, use_cache):
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    return None


def _write_cache(cache_path, df):
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
    return df


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, loader())


async def cached_async(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Same as cached() for a loader that returns a coroutine (the EFD client is async on the RSP)."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, await loader())


def zenith_500nm_factor(airmass, band):
    """Multiply an observed FWHM by this to refer it to zenith and 500 nm: X^-0.6 (lambda_eff / 500 nm)^0.2."""
    X = pd.to_numeric(pd.Series(airmass), errors="coerce").to_numpy(dtype=float)
    wavelength = (
        pd.Series(band)
        .astype("string")
        .str.strip()
        .str[0]
        .map(EFFECTIVE_WAVELENGTH_NM)
        .astype(float)
        .to_numpy()
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            np.where(X >= 1.0, X**-0.6, np.nan)
            * (wavelength / REFERENCE_WAVELENGTH_NM) ** 0.2
        )


def fried_parameter(seeing_arcsec, wavelength_nm=REFERENCE_WAVELENGTH_NM):
    """r0 (m) from a Kolmogorov FWHM in arcsec: r0 = 0.98 lambda / FWHM."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            0.98
            * wavelength_nm
            * 1e-9
            / (np.asarray(seeing_arcsec, dtype=float) * ARCSEC_TO_RAD)
        )


def outer_scale_factor(r0_m, L0):
    """sqrt(1 - 2.18 (r0/L0)^0.356) (Tokovinin 2002); 1 for L0=None; NaN where the bracket is not positive."""
    r0 = np.asarray(r0_m, dtype=float)
    if L0 is None:
        return np.where(np.isfinite(r0), 1.0, np.nan)
    with np.errstate(invalid="ignore"):
        bracket = 1.0 - 2.18 * np.power(r0 / float(L0), 0.356)
        return np.where(bracket > 0, np.sqrt(np.clip(bracket, 0, None)), np.nan)


def von_karman_seeing(r0_m, wavelength_nm, L0):
    """FWHM (arcsec) at wavelength_nm for a Fried parameter r0_m (m) at that wavelength and outer scale L0."""
    r0 = np.asarray(r0_m, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        kolmogorov = (
            0.98 * np.asarray(wavelength_nm, dtype=float) * 1e-9 / r0 / ARCSEC_TO_RAD
        )
    return kolmogorov * outer_scale_factor(r0, L0)


def dimm_seeing_in_visit_frame(dimm_seeing, airmass, band, L0):
    """DIMM (Kolmogorov, zenith / 500 nm) -> von Karman FWHM at the visit airmass and band, r0 scaled first:
    r0(lambda, X) = r0 (lambda_eff / 500 nm)^1.2 X^-0.6."""
    X = pd.to_numeric(pd.Series(airmass), errors="coerce").to_numpy(dtype=float)
    wavelength = (
        pd.Series(band)
        .astype("string")
        .str.strip()
        .str[0]
        .map(EFFECTIVE_WAVELENGTH_NM)
        .astype(float)
        .to_numpy()
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        r0 = (
            fried_parameter(dimm_seeing)
            * (wavelength / REFERENCE_WAVELENGTH_NM) ** 1.2
            * X**-0.6
        )
    return np.where(X >= 1.0, von_karman_seeing(r0, wavelength, L0), np.nan)


def signed_sqrt(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.sqrt(np.abs(x))

## ConsDB access

`visit1` (pointing, band, DIMM) joined to `visit1_quicklook` (PSF, guider seeing) and `exposure`
(`observation_reason`), one row per science visit, chunked by calendar month, through
`lsst.summit.utils.ConsDbClient` from the kernel image (the in-cluster USDF endpoint, or `LSST_CONSDB_PQ_URL` when set).
`fetch_visit_efd` reads the per-visit RINGSS mean from the transformed-EFD table `efd_lsstcam.visit1_efd`
(used when `ringss_source = consdb`, or if the EFD pull fails).

In [ ]:
USDF_CONSDB_URL = (
    "http://consdb-pq.consdb:8080/consdb"  # in-cluster (Times Square, USDF Nublado)
)
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "exp_time",
    "exp_midpt",
    "airmass",
    "zenith_distance",
    "altitude",
    "azimuth",
    "wind_speed",
    "dimm_seeing",
)
QUICKLOOK_COLUMNS = (
    "psf_sigma_median",
    "n_psf_star_total",
    "guider_mid_layer_seeing",
    "guider_free_seeing",
    "guider_ground_layer_seeing",
    "guider_total_seeing",
)
EXPOSURE_COLUMNS = ("observation_reason",)
VISIT_EFD_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "created_at",
    "mt_ringss_fwhm_scintillation",
)
TEXT_COLUMNS = (
    "band",
    "physical_filter",
    "science_program",
    "observation_reason",
    "img_type",
    "exp_midpt",
    "created_at",
)


class ConsDb:
    """lsst.summit.utils.ConsDbClient with .query(sql) -> DataFrame."""

    def __init__(self, url=None):
        from lsst.summit.utils import ConsDbClient

        self.url = url or os.environ.get("LSST_CONSDB_PQ_URL") or USDF_CONSDB_URL
        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(self.url)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    client = ConsDb()
    print(f"ConsDB client: lsst.summit.utils.ConsDbClient at {client.url}")
    return client


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def _numeric(df):
    for c in df.columns:
        if c not in TEXT_COLUMNS and df[c].dtype == object:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def build_query(
    day_obs_min, day_obs_max, programs, instrument="lsstcam", img_types=("science",)
):
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(client, day_obs_min, day_obs_max, programs, chunk_days=31):
    cols = list(VISIT_COLUMNS) + list(QUICKLOOK_COLUMNS) + list(EXPOSURE_COLUMNS)
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        if not set(cols).issubset(df.columns):
            raise RuntimeError(
                f"ConsDB returned an invalid schema for {lo}-{hi}; refusing partial results"
            )
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=cols)
    out = _numeric(pd.concat(frames, ignore_index=True))
    return out.sort_values(["day_obs", "seq_num"]).reset_index(drop=True)


def fetch_visit_efd(
    client, day_obs_min, day_obs_max, instrument="lsstcam", chunk_days=31
):
    """Per-visit RINGSS scintillation mean from efd_<instrument>.visit1_efd."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(
            "SELECT "
            + ", ".join(VISIT_EFD_COLUMNS)
            + f" FROM efd_{instrument}.visit1_efd"
            + f" WHERE day_obs >= {int(lo)} AND day_obs <= {int(hi)}"
        )
        print(f"  visit1_efd {lo}-{hi}: {len(df):6d} rows")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=list(VISIT_EFD_COLUMNS))
    out = _numeric(pd.concat(frames, ignore_index=True))
    return out.sort_values(["day_obs", "seq_num"]).reset_index(drop=True)

## EFD access: RINGSS

With `ringss_source = efd`, `fetch_ringss` pulls every `logevent_ringssMeasurement` (salIndex 304) over the night
range with `lsst_efd_client`, and `attach_ringss` gives each visit the median `fwhmScintillation` within
± `ringss_window_minutes` of the exposure midpoint (`ringss_z500`) and the number of measurements used (`ringss_n`).
If the EFD cannot be reached or returns nothing, the ConsDB `visit1_efd` mean is used instead, with a warning;
the source actually used is recorded in the Provenance cell.

In [ ]:
RINGSS_TOPIC = "lsst.sal.ESS.logevent_ringssMeasurement"
RINGSS_SAL_INDEX = 304
RINGSS_FIELD = "fwhmScintillation"


def get_efd_client(efd_name="usdf_efd"):
    import lsst_efd_client

    return lsst_efd_client.EfdClient(efd_name, output_mode="dataframe")


async def fetch_ringss(efd_client, day_obs_min, day_obs_max, chunk_days=31):
    """All RINGSS scintillation measurements from 12:00 UTC on day_obs_min to 12:00 UTC after day_obs_max."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        start, end = day_obs_window(lo, hi)
        df = efd_client.select_time_series(
            RINGSS_TOPIC, [RINGSS_FIELD], start, end, index=RINGSS_SAL_INDEX
        )
        if inspect.isawaitable(df):
            df = await df
        print(f"  ringss {lo}-{hi}: {len(df):6d} measurements")
        if len(df):
            df = df.reset_index().rename(columns={df.index.name or "index": "time"})
            frames.append(df[["time", RINGSS_FIELD]])
    if not frames:
        return pd.DataFrame(columns=["time", RINGSS_FIELD])
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"], utc=True).dt.tz_localize(None)
    out[RINGSS_FIELD] = pd.to_numeric(out[RINGSS_FIELD], errors="coerce")
    return out.dropna().sort_values("time").reset_index(drop=True)


async def load_ringss(
    source, consdb, efd_name, day_obs_min, day_obs_max, cache_dir, use_cache
):
    """(source actually used, table): EFD measurements for "efd", visit1_efd rows for "consdb"."""
    if source == "efd":
        try:
            efd = get_efd_client(efd_name)
            ringss = await cached_async(
                cache_dir
                and os.path.join(
                    cache_dir, f"ringss_efd_{day_obs_min}_{day_obs_max}.parquet"
                ),
                lambda: fetch_ringss(efd, day_obs_min, day_obs_max),
                use_cache,
            )
            if len(ringss):
                return "efd", ringss
            print(
                "WARNING: the EFD returned no RINGSS measurements; using ConsDB visit1_efd instead."
            )
        except Exception as exc:
            print(
                f"WARNING: RINGSS pull from the EFD failed ({exc!r}); using ConsDB visit1_efd instead."
            )
    visit_efd = cached(
        cache_dir
        and os.path.join(cache_dir, f"visit1_efd_{day_obs_min}_{day_obs_max}.parquet"),
        lambda: fetch_visit_efd(consdb, day_obs_min, day_obs_max),
        use_cache,
    )
    return "consdb", visit_efd


def attach_ringss(visits, source, table, window_minutes):
    """Add ringss_z500 (zenith / 500 nm scintillation seeing) and ringss_n to a copy of visits."""
    out = visits.copy()
    if source == "consdb":
        efd = table.drop_duplicates("visit_id")[
            ["visit_id", "mt_ringss_fwhm_scintillation"]
        ]
        out = out.merge(efd, on="visit_id", how="left")
        out["ringss_z500"] = pd.to_numeric(
            out.pop("mt_ringss_fwhm_scintillation"), errors="coerce"
        )
        out["ringss_n"] = out["ringss_z500"].notna().astype(int)
        return out
    t_r = table["time"].to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    vals = table[RINGSS_FIELD].to_numpy(dtype=float)
    t_v = (
        pd.to_datetime(out["exp_midpt"], utc=True, errors="coerce")
        .dt.tz_localize(None)
        .to_numpy(dtype="datetime64[ns]")
        .astype("int64")
        / 1e9
    )
    w = window_minutes * 60.0
    lo = np.searchsorted(t_r, t_v - w, side="left")
    hi = np.searchsorted(t_r, t_v + w, side="right")
    med = np.full(len(out), np.nan)
    for i, (a, b) in enumerate(zip(lo, hi)):
        if b > a:
            med[i] = np.median(vals[a:b])
    out["ringss_z500"] = med
    out["ringss_n"] = np.maximum(hi - lo, 0)
    return out

## Data, FWHM, atmosphere and selection

The ConsDB pull, the RINGSS attachment, the per-visit FWHM and the three atmospheric estimates in the frame of
the visit (`atm_<monitor>`, with `<monitor>_zenith_500nm` alongside for reference; for the DIMM that reference is the
outer-scale-corrected value at zenith / 500 nm, while `atm_dimm` comes from the r0 scaled to the visit), then `sys2_<monitor>` = FWHM² − atm².
The DIMM value is blanked on visits below `dimm_min_wind_speed` or without a wind reading. The selection table is the
ROSP-2 one; the counts after it are the visits each monitor can use.

In [ ]:
start_dayobs, end_dayobs = parse_day_obs(start_dayobs), parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
min_psf_stars = int(min_psf_stars)
if min_psf_stars < 0:
    raise ValueError(f"min_psf_stars ({min_psf_stars}) must be >= 0")
dimm_outer_scale_m = parse_outer_scale(dimm_outer_scale_m)
dimm_min_wind_speed = float(dimm_min_wind_speed or 0)
ringss_source = str(ringss_source).strip().lower()
if ringss_source not in ("efd", "consdb"):
    raise ValueError(f"ringss_source must be 'efd' or 'consdb', not {ringss_source!r}")
ringss_window_minutes = float(ringss_window_minutes)
use_cache = parse_bool(use_cache)
MONITORS = make_monitors(dimm_outer_scale_m, dimm_min_wind_speed)
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)
cache_tag = f"v1_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs) or 'all'}"
print(
    f"dayObs {start_dayobs} to {end_dayobs}; programs {programs or 'all'}; "
    f"min_psf_stars {min_psf_stars}; DIMM L0 {dimm_outer_scale_m}, wind >= {dimm_min_wind_speed:g} m/s; "
    f"RINGSS from {ringss_source}"
)

consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
ringss_source_used, ringss_table = await load_ringss(
    ringss_source, consdb, efd_name, start_dayobs, end_dayobs, cache_dir, use_cache
)
visits = attach_ringss(raw, ringss_source_used, ringss_table, ringss_window_minutes)

visits["band"] = visits["band"].astype("string").str.strip().str[0]
visits["fwhm"] = visits["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
visits["time"] = pd.to_datetime(
    visits["exp_midpt"], utc=True, errors="coerce"
).dt.tz_localize(None)
to_zenith_500nm = zenith_500nm_factor(visits["airmass"], visits["band"])
dimm = visits["dimm_seeing"].astype(float)
# reference column: the outer-scale-corrected DIMM at zenith / 500 nm (not what is subtracted)
visits["dimm_zenith_500nm"] = von_karman_seeing(
    fried_parameter(dimm), REFERENCE_WAVELENGTH_NM, dimm_outer_scale_m
)
# what is subtracted: r0 scaled to the band and airmass of the visit, then the von Karman FWHM there
visits["dimm_obs"] = dimm_seeing_in_visit_frame(
    dimm, visits["airmass"], visits["band"], dimm_outer_scale_m
)
if (
    dimm_min_wind_speed > 0
):  # the DIMM is unreliable in low wind: blank it there (and where the wind is unknown)
    windy = visits["wind_speed"].astype(float) >= dimm_min_wind_speed
    visits.loc[~windy.fillna(False), ["dimm_zenith_500nm", "dimm_obs"]] = np.nan
visits["guider_free_atm"] = np.sqrt(
    visits["guider_mid_layer_seeing"].astype(float) ** 2
    + visits["guider_free_seeing"].astype(float) ** 2
)
for m in MONITORS:
    value = visits[m.source].astype(float)
    if m.frame == "zenith_500nm":
        visits[m.z500] = value
        visits[m.atm] = value / to_zenith_500nm
    else:
        visits[m.atm] = value
        if (
            m.z500 not in visits
        ):  # the DIMM's zenith / 500 nm reference is computed above
            visits[m.z500] = value * to_zenith_500nm
    visits[m.sys2] = visits["fwhm"] ** 2 - visits[m.atm] ** 2

steps = [
    ("has PSF FWHM", visits["fwhm"].notna()),
    (
        f"n_psf_star_total >= {min_psf_stars}",
        visits["n_psf_star_total"].astype(float) >= min_psf_stars,
    ),
    (
        "has airmass >= 1 and a known band",
        (visits["airmass"] >= 1) & visits["band"].isin(BANDS),
    ),
    (
        f"observation_reason not in {EXCLUDE_REASONS}",
        ~visits["observation_reason"].isin(EXCLUDE_REASONS),
    ),
]
keep = pd.Series(True, index=visits.index)
rows = []
for name, mask in steps:
    mask = mask.fillna(False).astype(bool)
    rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
    keep &= mask
print(f"\nStart: {len(visits)} visits")
print(
    pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(index=False)
)
sel = visits[keep].copy()
for m in MONITORS:
    valid = np.isfinite(sel[m.atm]) & (sel[m.atm] > 0)
    sel.loc[~valid, [m.z500, m.atm, m.sys2]] = np.nan
print(
    f"\n{len(sel)} visits on {sel['day_obs'].nunique()} nights; with a usable monitor value:"
)
print(
    pd.DataFrame(
        {
            m.label: {
                "visits": int(sel[m.sys2].notna().sum()),
                "fraction": sel[m.sys2].notna().mean(),
                "nights": int(sel.loc[sel[m.sys2].notna(), "day_obs"].nunique()),
            }
            for m in MONITORS
        }
    ).T.round(3)
)
print()
print(
    pd.crosstab(sel["science_program"], sel["band"], margins=True).reindex(
        columns=[*BANDS, "All"], fill_value=0
    )
)

In [ ]:
# Sanity check on the de-correction: the factor applied to a zenith / 500 nm seeing for each band at a few airmasses.
probe = pd.DataFrame(
    {"airmass": np.repeat([1.0, 1.2, 1.5, 2.0], len(BANDS)), "band": list(BANDS) * 4}
)
probe["factor"] = 1.0 / zenith_500nm_factor(probe["airmass"], probe["band"])
probe.pivot(index="airmass", columns="band", values="factor")[list(BANDS)].round(3)

## Results

### Aggregate table

One row per monitor: the sDIQ (arcsec) with its error, the number of visits and nights behind it, the medians of
the ingredients (delivered FWHM, the monitor seeing in the frame of the visit and at zenith / 500 nm), and the
median, 25th and 75th percentiles of the per-visit $\mathrm{sys}^2$ (arcsec²) with the fraction of visits where it is negative.

In [ ]:
def sdiq_stats(sys2):
    """Median of sys2 (arcsec^2) with its robust standard error, the quartiles, and the signed-sqrt sDIQ."""
    s = np.asarray(sys2, dtype=float)
    s = s[np.isfinite(s)]
    keys = (
        "n",
        "sdiq",
        "sdiq_err",
        "median_sys2",
        "median_sys2_err",
        "p25_sys2",
        "p75_sys2",
        "frac_negative",
    )
    if not len(s):
        return dict.fromkeys(keys, np.nan) | {"n": 0}
    med = np.median(s)
    err = 1.2533 * 1.4826 * np.median(np.abs(s - med)) / np.sqrt(len(s))
    return dict(
        n=len(s),
        sdiq=float(signed_sqrt(med)),
        sdiq_err=err / (2.0 * np.sqrt(abs(med))) if med != 0 else np.nan,
        median_sys2=med,
        median_sys2_err=err,
        p25_sys2=np.percentile(s, 25),
        p75_sys2=np.percentile(s, 75),
        frac_negative=float((s < 0).mean()),
    )


def aggregate_table(df, monitors):
    rows = []
    for m in monitors:
        ok = df[m.sys2].notna()
        st = sdiq_stats(df.loc[ok, m.sys2])
        rows.append(
            {
                "monitor": m.label,
                "sDIQ": st["sdiq"],
                "sDIQ err": st["sdiq_err"],
                "n_visits": st["n"],
                "n_nights": int(df.loc[ok, "day_obs"].nunique()),
                "median FWHM": df.loc[ok, "fwhm"].median(),
                "median atm (visit frame)": df.loc[ok, m.atm].median(),
                "median atm (zenith/500nm)": df.loc[ok, m.z500].median(),
                "median airmass": df.loc[ok, "airmass"].median(),
                "sys2 p25": st["p25_sys2"],
                "sys2 median": st["median_sys2"],
                "sys2 p75": st["p75_sys2"],
                "frac sys2 < 0": st["frac_negative"],
            }
        )
    return pd.DataFrame(rows).set_index("monitor")


summary = aggregate_table(sel, MONITORS)
for m in MONITORS:
    row = summary.loc[m.label]
    print(
        f'{m.label}: sDIQ = {row["sDIQ"]:.3f} +/- {row["sDIQ err"]:.3f} arcsec  '
        f'(median sys2 {row["sys2 median"]:+.4f} arcsec2; n = {int(row["n_visits"])} visits on {int(row["n_nights"])} nights)'
    )
summary.round(3).T

### Per band

sDIQ per monitor for each band (and all bands). A band-dependent sDIQ would point at the $\lambda^{-0.2}$ de-correction
of the monitors or at band-specific system terms.

In [ ]:
def band_table(df, monitors):
    groups = [("all", df)] + [
        (b, df[df["band"] == b]) for b in BANDS if (df["band"] == b).any()
    ]
    rows = []
    for name, g in groups:
        row = {"band": name, "median airmass": g["airmass"].median()}
        for m in monitors:
            st = sdiq_stats(g[m.sys2])
            row[f"sDIQ {m.tag}"] = st["sdiq"]
            row[f"err {m.tag}"] = st["sdiq_err"]
            row[f"n {m.tag}"] = st["n"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("band")


band_table(sel, MONITORS).round(3)

### Nightly time series

Top: the nightly median delivered FWHM and each monitor's seeing in the frame of the visit, with 25-75 % whiskers
over the visits of the night. Bottom: the nightly sDIQ per monitor, the square root of that night's median
$\mathrm{sys}^2$, with whiskers at the signed square roots of the night's 25th and 75th percentiles of the per-visit
$\mathrm{sys}^2$. Nights with fewer than 5 usable visits for a monitor are left out for that monitor; series are offset slightly in x.

In [ ]:
def nightly_table(df, monitors, min_visits=MIN_VISITS_PER_NIGHT):
    def q(x, p):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return np.percentile(x, p) if len(x) else np.nan

    rows = []
    for day_obs, sub in df.groupby("day_obs"):
        row = dict(
            day_obs=int(day_obs),
            date=pd.Timestamp(str(int(day_obs))),
            n_visits=len(sub),
            fwhm=q(sub["fwhm"], 50),
            fwhm_p25=q(sub["fwhm"], 25),
            fwhm_p75=q(sub["fwhm"], 75),
        )
        for m in monitors:
            st = sdiq_stats(sub[m.sys2])
            ok = st["n"] >= min_visits
            row[f"n_{m.tag}"] = st["n"]
            row[m.atm] = q(sub[m.atm], 50) if ok else np.nan
            row[f"{m.atm}_p25"] = q(sub[m.atm], 25) if ok else np.nan
            row[f"{m.atm}_p75"] = q(sub[m.atm], 75) if ok else np.nan
            row[f"sdiq_{m.tag}"] = st["sdiq"] if ok else np.nan
            row[f"sdiq_{m.tag}_p25"] = (
                float(signed_sqrt(st["p25_sys2"])) if ok else np.nan
            )
            row[f"sdiq_{m.tag}_p75"] = (
                float(signed_sqrt(st["p75_sys2"])) if ok else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows)


def whiskers(ax, nightly, col, color, marker, label, offset_days=0.0, ms=4):
    x = nightly["date"] + pd.Timedelta(days=offset_days)
    ax.errorbar(
        x,
        nightly[col],
        yerr=np.array(
            [nightly[col] - nightly[f"{col}_p25"], nightly[f"{col}_p75"] - nightly[col]]
        ),
        fmt=marker,
        ms=ms,
        color=color,
        ecolor=color,
        elinewidth=1.0,
        capsize=0,
        alpha=0.9,
        label=label,
    )


nightly = nightly_table(sel, MONITORS)
offsets = np.linspace(-0.2, 0.2, len(MONITORS) + 1)
fig, axes = plt.subplots(
    2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [1, 1.3]}
)
ax = axes[0]
whiskers(ax, nightly, "fwhm", INK, "o", "delivered PSF FWHM", offsets[0])
for k, m in enumerate(MONITORS):
    whiskers(
        ax,
        nightly,
        m.atm,
        m.color,
        m.marker,
        f"{m.label} (visit frame)",
        offsets[k + 1],
    )
ax.set_ylabel("per-visit value (arcsec)")
ax.set_title(
    f"{len(sel)} visits on {sel['day_obs'].nunique()} nights", loc="left", fontsize=11
)
ax.legend(
    ncol=2,
    fontsize=9,
    title="marker: nightly median; whisker: 25-75 %",
    title_fontsize=9,
)
ax = axes[1]
for k, m in enumerate(MONITORS):
    whiskers(ax, nightly, f"sdiq_{m.tag}", m.color, m.marker, m.label, offsets[k + 1])
ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
ax.set_ylabel("sDIQ = sqrt(median FWHM² - atm²) (arcsec)")
ax.set_xlabel("night (day_obs)")
ax.legend(
    ncol=3,
    fontsize=9,
    title="marker: nightly sDIQ; whisker: signed sqrt of the 25-75 % range of per-visit FWHM² - atm²",
    title_fontsize=9,
)
fig.align_ylabels(axes)
plt.show()

nightly.set_index("day_obs").drop(columns="date").round(3)

### Histograms

Per-visit $\mathrm{sys}^2$ = FWHM² − atm² for each monitor, as a fraction of that monitor's visits per bin, with the
median (solid) and 25 / 75 % (dotted) of each marked above the curves. The legend quotes the sDIQ (the square root of the median) and the three values.

In [ ]:
bins = np.arange(-1.0, 2.0 + 0.025, 0.025)
fig, ax = plt.subplots(figsize=(11, 6.5))
ymax = 0
stats = {}
for m in MONITORS:
    d = sel[m.sys2].dropna()
    if not len(d):
        continue
    st = stats[m.tag] = sdiq_stats(d)
    counts, _, _ = ax.hist(
        d.clip(bins[0], bins[-1]),
        bins=bins,
        histtype="step",
        lw=2,
        color=m.color,
        weights=np.full(len(d), 1 / len(d)),
        label=(
            f'{m.label}: sDIQ {st["sdiq"]:.3f}"; sys² {st["median_sys2"]:+.3f} '
            f'[{st["p25_sys2"]:+.3f}, {st["p75_sys2"]:+.3f}] arcsec²  (n = {st["n"]})'
        ),
    )
    ymax = max(ymax, counts.max())
for k, m in enumerate(MONITORS):
    if m.tag not in stats:
        continue
    st = stats[m.tag]
    y = ymax * (1.06 + 0.06 * k)
    ax.plot([st["p25_sys2"], st["p75_sys2"]], [y, y], color=m.color, lw=1.2, ls=":")
    ax.plot([st["median_sys2"]], [y], marker="|", ms=12, mew=2, color=m.color)
    ax.axvline(st["median_sys2"], color=m.color, lw=1.0, ls="-", alpha=0.6)
ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
ax.set_ylim(0, ymax * (1.06 + 0.06 * len(MONITORS) + 0.04))
ax.set_xlim(bins[0], bins[-1])
ax.set_xlabel("PSF FWHM² - atmosphere² (arcsec²)")
ax.set_ylabel("fraction of visits per bin")
ax.set_title(f"{start_dayobs}-{end_dayobs}, {', '.join(programs) or 'all programs'}")
ax.legend(
    fontsize=9,
    title="sDIQ; median [25 %, 75 %] of sys²",
    title_fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.13),
)
plt.show()

### Sanity checks

**sDIQ against elevation.** One panel per monitor: the sDIQ (square root of the binned median of $\mathrm{sys}^2$, error bar from the
robust error on the median) in bins of telescope altitude, for all bands (black) and per band, with the visits per bin underneath.
The atmosphere is removed in the frame of the visit, so a residual trend with elevation would point at the $X^{0.6}$
scaling of a monitor or at a real elevation dependence of the system (mirror support, wind exposure). Bins with fewer than 10 visits are skipped.

In [ ]:
MIN_VISITS_PER_BIN = 10


def binned_sdiq(x, sys2, edges, min_count=MIN_VISITS_PER_BIN):
    """sdiq_stats of sys2 in bins of x; bins with fewer than min_count visits are left NaN."""
    x, s = np.asarray(x, dtype=float), np.asarray(sys2, dtype=float)
    ok = np.isfinite(x) & np.isfinite(s)
    idx = np.digitize(x[ok], edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        st = sdiq_stats(s[ok][idx == i])
        if st["n"] < min_count:
            st = dict.fromkeys(st, np.nan) | {"n": st["n"]}
        rows.append({"center": 0.5 * (edges[i] + edges[i + 1]), **st})
    return pd.DataFrame(rows)


alt_edges = np.arange(20.0, 90.1, 5.0)
groups = [("all", INK, sel)] + [
    (b, BAND_COLORS[b], sel[sel["band"] == b])
    for b in BANDS
    if (sel["band"] == b).any()
]
fig, axes = plt.subplots(
    2,
    len(MONITORS),
    figsize=(5.6 * len(MONITORS), 8),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
    squeeze=False,
)
for k, m in enumerate(MONITORS):
    ax, ax_n = axes[0, k], axes[1, k]
    for name, color, g in groups:
        st = binned_sdiq(g["altitude"], g[m.sys2], alt_edges)
        if st["sdiq"].notna().any():
            ax.errorbar(
                st["center"],
                st["sdiq"],
                yerr=st["sdiq_err"],
                marker="o" if name == "all" else "s",
                ms=5 if name == "all" else 3.5,
                lw=2 if name == "all" else 1.2,
                capsize=2,
                color=color,
                label="all bands" if name == "all" else f"{name} band",
            )
        d = g.loc[g[m.sys2].notna(), "altitude"].dropna()
        if len(d):
            ax_n.hist(
                d,
                bins=alt_edges,
                histtype="step",
                lw=2 if name == "all" else 1.2,
                color=color,
            )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_title(f"{m.label}  (n = {int(sel[m.sys2].notna().sum())})", fontsize=11)
    ax.set_ylabel("sDIQ = sqrt(binned median FWHM² - atm²) (arcsec)")
    ax_n.set_xlabel("telescope elevation (deg)")
    ax_n.set_ylabel("visits")
    ax_n.set_xlim(alt_edges[0], alt_edges[-1])
axes[0, 0].legend(fontsize=8.5, ncol=2)
fig.align_ylabels(axes[:, 0])
plt.show()

**Histograms per band.** The per-visit $\mathrm{sys}^2$ of each monitor split by band (fraction of that band's visits per bin), with the
median (tick) and 25 / 75 % (dotted) of each band marked above the curves and the all-band median as the vertical line. The legend
quotes the per-band sDIQ. A band whose distribution sits apart from the others has either a chromatic system term or a monitor scaling problem.

In [ ]:
fig, axes = plt.subplots(
    1, len(MONITORS), figsize=(6 * len(MONITORS), 7.5), sharey=True, squeeze=False
)
for ax, m in zip(axes[0], MONITORS):
    ymax, stats = 0, {}
    for b in BANDS:
        d = sel.loc[sel["band"] == b, m.sys2].dropna()
        if len(d) < 2:
            continue
        st = stats[b] = sdiq_stats(d)
        counts, _, _ = ax.hist(
            d.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=1.8,
            color=BAND_COLORS[b],
            weights=np.full(len(d), 1 / len(d)),
            label=f'{b}: sDIQ {st["sdiq"]:.3f} +/- {st["sdiq_err"]:.3f}"  (n = {st["n"]})',
        )
        ymax = max(ymax, counts.max())
    for k, (b, st) in enumerate(stats.items()):
        y = ymax * (1.06 + 0.05 * k)
        ax.plot(
            [st["p25_sys2"], st["p75_sys2"]],
            [y, y],
            color=BAND_COLORS[b],
            lw=1.2,
            ls=":",
        )
        ax.plot(
            [st["median_sys2"]], [y], marker="|", ms=10, mew=2, color=BAND_COLORS[b]
        )
    st_all = sdiq_stats(sel[m.sys2])
    ax.axvline(
        st_all["median_sys2"],
        color=INK,
        lw=1.2,
        label=f'all bands: sDIQ {st_all["sdiq"]:.3f}"',
    )
    ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_ylim(0, ymax * (1.06 + 0.05 * len(stats) + 0.04) if ymax else 1)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_xlabel("PSF FWHM² - atmosphere² (arcsec²)")
    ax.set_title(m.label, fontsize=11)
    ax.legend(
        fontsize=8.5,
        title="median [25 %, 75 %] ticks above the curves",
        title_fontsize=8.5,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
    )
axes[0, 0].set_ylabel("fraction of the band's visits per bin")
plt.show()

## Provenance

What this run was computed with: the method version, the pinned inputs (PSF column, the three monitors and
the RINGSS source actually used, the DIMM outer scale), the kernel image and the library versions. The effective
wavelengths hard-coded above are compared with the `lsst.summit.utils` installed in this kernel, using the physical
filters present in the selection; a mismatch means summit_utils changed and the Method Change Process decides whether the notebook follows.

In [ ]:
import platform
import shutil
import subprocess


def check_effective_wavelengths(df):
    """Compare EFFECTIVE_WAVELENGTH_NM with the summit_utils of this kernel, per physical filter in df."""
    try:
        from lsst.summit.utils.utils import getBandpassSeeingCorrection
    except ImportError:
        return "not checked (lsst.summit.utils not importable in this kernel)"
    problems = []
    pairs = (
        df[["band", "physical_filter"]]
        .dropna()
        .drop_duplicates()
        .itertuples(index=False)
    )
    for band, physical_filter in pairs:
        try:
            installed_nm = (
                500.0 * getBandpassSeeingCorrection(str(physical_filter)) ** 5
            )
        except ValueError as exc:
            problems.append(f"{physical_filter}: {exc}")
            continue
        if abs(installed_nm - EFFECTIVE_WAVELENGTH_NM[band]) > 0.01:
            problems.append(
                f"{physical_filter}: notebook {EFFECTIVE_WAVELENGTH_NM[band]:.4f} nm, "
                f"installed {installed_nm:.4f} nm"
            )
    return "match" if not problems else "MISMATCH - " + "; ".join(problems)


def stack_version():
    out = []
    try:
        from lsst.summit.utils import version as su_version

        out.append(f"summit_utils {su_version.__version__}")
    except Exception:
        out.append("summit_utils not importable")
    if shutil.which("eups"):
        res = subprocess.run(
            ["eups", "list", "-s", "lsst_distrib"], capture_output=True, text=True
        )
        out.append(f"lsst_distrib {' '.join(res.stdout.split()) or 'not set up'}")
    return "; ".join(out)


ringss_description = (
    f"EFD {RINGSS_TOPIC} salIndex {RINGSS_SAL_INDEX} {RINGSS_FIELD}, median within +/- {ringss_window_minutes:g} min "
    f"of the visit midpoint ({efd_name})"
    if ringss_source_used == "efd"
    else "ConsDB efd_lsstcam.visit1_efd.mt_ringss_fwhm_scintillation (per-visit mean)"
)
provenance = {
    "method version": METHOD_VERSION,
    "run (UTC)": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "PSF column": "cdb_lsstcam.visit1_quicklook.psf_sigma_median (Summit Rapid Analysis; "
    "producer version not recorded in ConsDB), as delivered",
    "RINGSS": ringss_description
    + (
        ""
        if ringss_source_used == ringss_source
        else f" (requested {ringss_source}, fell back)"
    ),
    "DIMM": "cdb_lsstcam.visit1.dimm_seeing, von Karman outer-scale correction (Tokovinin 2002) "
    + (f"L0 = {dimm_outer_scale_m:g} m" if dimm_outer_scale_m else "off")
    + ", evaluated at the visit band and airmass (r0 scaled by (lambda/500)^1.2 X^-0.6 first)"
    + (
        f"; only visits with visit1.wind_speed >= {dimm_min_wind_speed:g} m/s"
        if dimm_min_wind_speed > 0
        else "; no wind-speed cut"
    ),
    "guider": "sqrt(guider_mid_layer_seeing^2 + guider_free_seeing^2) from cdb_lsstcam.visit1_quicklook "
    f"(lsst.summit.utils.guiders.seeing), frame: {GUIDER_SEEING_FRAME}",
    "monitor frame": "RINGSS: X^0.6 (lambda_eff/500 nm)^-0.2; DIMM: through r0 (above); guider: as stored",
    "effective wavelengths": EFFECTIVE_WAVELENGTH_SOURCE,
    "wavelength check vs kernel": check_effective_wavelengths(sel),
    "ConsDB endpoint": consdb.url,
    "kernel image": os.environ.get("JUPYTER_IMAGE_SPEC", "not a Nublado pod"),
    "image description": os.environ.get("IMAGE_DESCRIPTION", "n/a"),
    "stack": stack_version(),
    "python / numpy / pandas": f"{platform.python_version()} / {np.__version__} / {pd.__version__}",
}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame.from_dict(provenance, orient="index", columns=["value"]))